In [2]:
# Day 3: Feature Engineering
# Goal: Create new columns that help with analysis

import pandas as pd
from datetime import datetime

# Load cleaned data
print("Loading cleaned data...")
df = pd.read_csv('customer_complaints_cleaned.csv')
print(f"Starting with {len(df)} rows\n")

print("="*60)
print("FEATURE 1: EXTRACT DATE COMPONENTS")
print("="*60)

# Convert date column to datetime type
df['date_submitted'] = pd.to_datetime(df['date_submitted'])

# Extract year, month, quarter, day of week
df['year'] = df['date_submitted'].dt.year
df['month'] = df['date_submitted'].dt.month
df['month_name'] = df['date_submitted'].dt.month_name()
df['quarter'] = df['date_submitted'].dt.quarter
df['day_of_week'] = df['date_submitted'].dt.day_name()
df['week_of_year'] = df['date_submitted'].dt.isocalendar().week

print("New date columns created:")
print(df[['date_submitted', 'year', 'month', 'month_name', 'quarter', 'day_of_week']].head())
print()

print("="*60)
print("FEATURE 2: CATEGORIZE COMPLAINTS BY TYPE")
print("="*60)

# Categorize complaints based on keywords in complaint text
def categorize_complaint(text):
    """Assign complaint category based on keywords"""
    text = str(text).lower()
    
    if any(word in text for word in ['delivery', 'delayed', 'shipping', 'late']):
        return 'delivery'
    elif any(word in text for word in ['broken', 'defective', 'not working', 'damaged', 'quality']):
        return 'product_quality'
    elif any(word in text for word in ['rude', 'unprofessional', 'staff', 'service', 'wait time']):
        return 'customer_service'
    elif any(word in text for word in ['refund', 'charged', 'bill', 'overcharged', 'payment']):
        return 'billing'
    elif any(word in text for word in ['website', 'crash', 'technical', 'error']):
        return 'technical'
    elif any(word in text for word in ['wrong item', 'missing']):
        return 'order_issue'
    else:
        return 'other'

df['complaint_category'] = df['complaint_text'].apply(categorize_complaint)

print("Complaint categories created:")
print(df['complaint_category'].value_counts())
print()

print("="*60)
print("FEATURE 3: RESOLUTION SPEED CATEGORIES")
print("="*60)

# Create resolution speed category
def categorize_resolution_speed(hours):
    """Categorize resolution time into speed buckets"""
    if pd.isna(hours):
        return 'unknown'
    elif hours <= 24:
        return 'fast'
    elif hours <= 72:
        return 'normal'
    else:
        return 'slow'

df['resolution_speed'] = df['resolution_time_hours'].apply(categorize_resolution_speed)

print("Resolution speed categories:")
print(df['resolution_speed'].value_counts())
print()

print("="*60)
print("FEATURE 4: CUSTOMER SEGMENTATION")
print("="*60)

# Calculate high-value customer threshold (top 20%)
value_threshold = df['lifetime_value'].quantile(0.80)
print(f"High-value customer threshold: ${value_threshold:.2f}")

# Flag high-value customers
df['is_high_value_customer'] = df['lifetime_value'] >= value_threshold
df['is_high_value_customer'] = df['is_high_value_customer'].fillna(False)

print(f"\nHigh-value customers: {df['is_high_value_customer'].sum()}")
print(f"Regular customers: {(~df['is_high_value_customer']).sum()}")
print()

# Categorize customer tenure
def categorize_tenure(months):
    """Categorize customer tenure"""
    if pd.isna(months):
        return 'unknown'
    elif months <= 6:
        return 'new'
    elif months <= 24:
        return 'regular'
    else:
        return 'loyal'

df['customer_segment'] = df['customer_tenure_months'].apply(categorize_tenure)

print("Customer segments by tenure:")
print(df['customer_segment'].value_counts())
print()

print("="*60)
print("FEATURE 5: REPEAT COMPLAINT FLAG")
print("="*60)

# Standardize repeat complaint to boolean
df['is_repeat_complaint'] = df['repeat_complaint'].isin(['yes', 'y', '1', 'true'])

print(f"Repeat complaints: {df['is_repeat_complaint'].sum()}")
print(f"First-time complaints: {(~df['is_repeat_complaint']).sum()}")
print()

print("="*60)
print("FEATURE 6: SATISFACTION CATEGORIES")
print("="*60)

# Categorize satisfaction scores
def categorize_satisfaction(score):
    """Categorize satisfaction into groups"""
    if pd.isna(score):
        return 'not_rated'
    elif score <= 2:
        return 'poor'
    elif score == 3:
        return 'neutral'
    else:
        return 'good'

df['satisfaction_category'] = df['satisfaction_score'].apply(categorize_satisfaction)

print("Satisfaction categories:")
print(df['satisfaction_category'].value_counts())
print()

print("="*60)
print("FEATURE 7: TIME-BASED FLAGS")
print("="*60)

# Flag weekend complaints
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday'])

# Flag holiday season (Nov-Dec)
df['is_holiday_season'] = df['month'].isin([11, 12])

print(f"Weekend complaints: {df['is_weekend'].sum()}")
print(f"Holiday season complaints: {df['is_holiday_season'].sum()}")
print()

print("="*60)
print("FEATURE ENGINEERING COMPLETE!")
print("="*60)

print(f"\nFinal dataset: {len(df)} rows, {len(df.columns)} columns")
print("\nNew columns added:")
new_columns = ['year', 'month', 'month_name', 'quarter', 'day_of_week', 
               'complaint_category', 'resolution_speed', 'is_high_value_customer',
               'customer_segment', 'is_repeat_complaint', 'satisfaction_category',
               'is_weekend', 'is_holiday_season']
print(new_columns)

# Show sample of enriched data
print("\nSample of enriched data:")
print(df[['complaint_id', 'complaint_category', 'resolution_speed', 
          'customer_segment', 'satisfaction_category']].head(10))

# Save enriched data
df.to_csv('customer_complaints_enriched.csv', index=False)
print("\n✅ Enriched data saved as: customer_complaints_enriched.csv")
print("\nReady for SQL export! 🚀")

Loading cleaned data...
Starting with 25000 rows

FEATURE 1: EXTRACT DATE COMPONENTS
New date columns created:
  date_submitted  year  month month_name  quarter day_of_week
0     2022-09-23  2022      9  September        3      Friday
1     2023-12-11  2023     12   December        4      Monday
2     2023-05-19  2023      5        May        2      Friday
3     2022-09-12  2022      9  September        3      Monday
4     2024-05-05  2024      5        May        2      Sunday

FEATURE 2: CATEGORIZE COMPLAINTS BY TYPE
Complaint categories created:
complaint_category
product_quality     8860
customer_service    7139
billing             2865
order_issue         2850
delivery            1827
technical           1459
Name: count, dtype: int64

FEATURE 3: RESOLUTION SPEED CATEGORIES
Resolution speed categories:
resolution_speed
slow       15186
normal      5771
fast        2776
unknown     1267
Name: count, dtype: int64

FEATURE 4: CUSTOMER SEGMENTATION
High-value customer threshold: $4015